In [5]:
import sys, json, datetime
from pathlib import Path
from xgboost import XGBClassifier

repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "test_001_nvda").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import common as rq

STRATEGY_DIR = repo_root / "backtesting" / "test_001_nvda"
config = rq.load_strategy_config(STRATEGY_DIR / "strategy_config.json")
model_path = rq.build_model_path(repo_root, "test_001_nvda", config["model_name"])

# --- Deprecated: basic XGBClassifier baseline ---
df = rq.fetch_ohlcv(
    config["stock_symbol"],
    start=config["backtest_start"],
    end=config["backtest_end"],
    provider=config["data_src"],
)
df["return"] = df["close"].pct_change()
df["vol_change"] = df["volume"].pct_change()
df["target"] = (df["return"].shift(-1) > 0).astype(int)

X = df[["return", "vol_change"]].dropna()[:-1]
y = df["target"].loc[X.index]

model = XGBClassifier(n_estimators=10, max_depth=3, learning_rate=0.1)
model.fit(X, y)
model.save_model(model_path)
print(f"Saved baseline model → {model_path}")

Saved baseline model → /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/test-001-nvda.json


In [6]:
FEE_BPS, GAMMA = 5, 0.95

df = rq.fetch_ohlcv(
    config["stock_symbol"],
    start=config["backtest_start"],
    end=config["backtest_end"],
    provider=config["data_src"],
)
df = rq.add_indicators(df).pipe(rq.add_gate_signals).dropna()
transitions = rq.build_transitions(df, transaction_cost=FEE_BPS / 10_000)

print(f"Bars: {len(df)}  ({df.index[0]} → {df.index[-1]})")
transitions["action_name"].value_counts()

Bars: 232  (2022-01-31 → 2022-12-30)


action_name
hold    460
sell      7
buy       4
Name: count, dtype: int64

In [7]:
action_models = rq.fitted_q_iteration(transitions, n_iter=8, gamma=GAMMA)

state_catalog = (
    transitions[["state_row", "date", *rq.STATE_COLUMNS, "buy_signal", "sell_signal"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
scores = rq.score_valid_actions(
    state_catalog[rq.STATE_COLUMNS],
    state_catalog["buy_signal"],
    state_catalog["sell_signal"],
    state_catalog["position_flag"],
    action_models,
)
state_catalog["chosen_action"] = scores.idxmax(axis=1)

state_catalog[["date", "position_flag", "buy_signal", "sell_signal", "chosen_action"]].head()

,date,position_flag,buy_signal,sell_signal,chosen_action
0,2022-01-31,0,0,0,hold
1,2022-01-31,1,0,0,hold
2,2022-02-01,0,0,0,hold
3,2022-02-01,1,0,0,hold
4,2022-02-02,0,0,0,hold


In [8]:
rl_model_paths = {}
for action_name, model in action_models.items():
    if model is None:
        continue
    artifact_path = model_path.parent / f"{config['model_name']}-q-{action_name}.json"
    model.save_model(artifact_path)
    rl_model_paths[action_name] = str(artifact_path)

metadata_path = model_path.parent / f"{config['model_name']}-policy-metadata.json"
metadata_path.write_text(json.dumps({
    "model_name": config["model_name"],
    "policy_type": "fitted_q_iteration",
    "actions": {"hold": 0, "buy": 1, "sell": 2},
    "state_columns": rq.STATE_COLUMNS,
    "gamma": GAMMA,
    "transaction_cost_bps": FEE_BPS,
    "indicator_parameters": {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "rsi_period": 14, "cci_period": 20},
    "gate_rules": {
        "buy": "macd_line crosses above macd_signal and rsi > 50 while flat",
        "sell": "macd_line crosses below macd_signal and rsi < 50 while long",
        "hold": "always valid",
    },
    "artifacts": rl_model_paths,
}, indent=2))

print(f"Saved {len(rl_model_paths)} models + metadata → {metadata_path.parent}/")

Saved 3 models + metadata → /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/
